## Definir agente

In [1]:
from langchain_openai import ChatOpenAI

# 1. Initialize the local LM Studio model
local_llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1",  # Points to LM Studio
    api_key="lm-studio",                 # LM Studio doesn't require a real key, but a placeholder string prevents errors
    model="google/gemma-4-e2b",      # Match the name/identifier loaded in LM Studio
    temperature=0                       # Low temperature is critical for agent tool accuracy
)

In [2]:
from deepagents import create_deep_agent
from pathlib import Path

# Importar api key desde las variables de entorno
from dotenv import load_dotenv
load_dotenv()

# importar tools (relative import for running from this directory)
from tools import download_viirs_snpp_peru, fires_clean, download_ctry_shp, intersect_and_collapse_fires_peru, folium_choropleth_peru

system_prompt = """You are a geospatial analyst for Peru wildfire data. 

When asked to analyze fires, load and follow the peru-fire-analysis skill.
Report statistics after aggregation and provide output file paths."""

# Get the directory where this script lives (for skills path)
SCRIPT_DIR = Path("/Users/jgastanaduy/GitHub/geo-agent/deepagents/clase2")

# Definir un agente con el modelo Gemini 3.5 Flash de Google GenAI
agent = create_deep_agent(
    model= local_llm,
    tools=[download_viirs_snpp_peru, fires_clean, download_ctry_shp, 
           intersect_and_collapse_fires_peru, folium_choropleth_peru],
    system_prompt=system_prompt,
    skills=[str(SCRIPT_DIR)]  # Points to folder with SKILL.md
)

## Probar agente

In [3]:
query = """ Download and analyze forestal fires in Peru for 2024.
I want to see which departments had the most fires and generate an HTML map."""

In [4]:
# Run the agent
result = agent.invoke(
        {"messages": [{"role": "user", 
                       "content": query}]}
)

In [5]:
# Usar librería rich para imprimir el resultado de manera más legible
from rich import print as rprint
rprint(result)

{
    'messages': [
        HumanMessage(
            content=' Download and analyze forestal fires in Peru for 2024.\nI want to see which departments had 
the most fires and generate an HTML map.',
            additional_kwargs={},
            response_metadata={},
            id='a174e868-f3d1-4726-b1f9-89efc35ab53d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 423,
                    'prompt_tokens': 6863,
                    'total_tokens': 7286,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 297,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': None
                },
                'model_provider': 'openai',
                'model_name': 'google/gemma-4-e2b',
                'system_fingerprint': 'google/gemma-4-e2b',
                'id': 'chatcmpl-ev2mycz6008g1z47gi0dp',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f72ff-c074-7830-aa6b-30d143d433b8-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {
                                'content': 'Download VIIRS fires data for Peru for the year 2024.',
                                'status': 'in_progress'
                            },
                            {'content': 'Clean the downloaded fire data.', 'status': 'pending'},
                            {
                                'content': 'Download Peru administrative boundary polygons (departments).',
                                'status': 'pending'
                            },
                            {
                                'content': 'Intersect fire points with department polygons and aggregate fire 
counts.',
                                'status': 'pending'
                            },
                            {
                                'content': 'Generate an HTML choropleth map of fire counts by department.',
                                'status': 'pending'
                            }
                        ]
                    },
                    'id': '907900666',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 6863,
                'output_tokens': 423,
                'total_tokens': 7286,
                'input_token_details': {},
                'output_token_details': {'reasoning': 297}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': 'Download VIIRS fires data for Peru for the year 2024.', 
'status': 'in_progress'}, {'content': 'Clean the downloaded fire data.', 'status': 'pending'}, {'content': 
'Download Peru administrative boundary polygons (departments).', 'status': 'pending'}, {'content': 'Intersect fire 
points with department polygons and aggregate fire counts.', 'status': 'pending'}, {'content': 'Generate an HTML 
choropleth map of fire counts by department.', 'status': 'pending'}]",
            name='write_todos',
            id='3f827b57-4fc8-46ae-9d12-c96354233ac2',
            tool_call_id='907900666'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 7110,
                    'total_tokens': 7133,
                    'completion_tokens_details': {
                        'acce